# Finetune Vintern-1B-v3_5 voi QLoRA tren Kaggle

**Mục tiêu Notebook**: Notebook Kaggle sẵn sàng chạy để finetune model Vintern-1B-v3_5 sử dụng kỹ thậut QLoRA (4-bit quantization + LoRA) tren dataset NCKH vintern.

**Cac tinh nang chinh**:
- Hỗ trợ QLoRA (memory efficient, 4-bit quantization)
- Gradient checkpointing được kích hoạt
- Script huyến luyện đã được custom lại từ bản gốc của Vintern và upload thành Dataset của Kaggle để import ngược vào sử dụng (phiên bản từ `/kaggle/input/datasets/asiscd/vintern-finetune/`)
- Tu dong tao file meta voi cac truong length cua dataset

In [ ]:
# ====================== PHAN 1: CAI DAT CAC THU VIEN ======================
import subprocess
import sys

print("Dang cai dat cac goi can thiet...")

packages = [
    "timm",
    "einops", 
    "peft",
    "wandb",
    "deepspeed",
    "accelerate",
    "bitsandbytes",
    "decord",
    "tensorboardX",
    "gdown"
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Hoan thanh cai dat cac goi co ban")

# Nang cap cac goi quan trong
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "datasets"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers==4.47.0"])

print("Hoan thanh cai dat tat ca thu vien!")

In [ ]:
# ====================== PHAN 2: NHAN BAN KHO LUU TRU INTERNVL ======================
import os
import subprocess

print("Dang nhan ban kho luu tru InternVL...")

if not os.path.exists("/kaggle/working/Vintern"):
    os.chdir("/kaggle/working")
    subprocess.run(["git", "clone", "https://github.com/5CD-AI/Vintern.git"], check=True)
    print("Hoan thanh nhan ban kho luu tru")
else:
    print("Kho luu tru da ton tai")

os.chdir("/kaggle/working/Vintern")
print(f"Vi tri: {os.getcwd()}")

In [ ]:
# ====================== PHAN 2.5: CAU HINH WANDB LOGGING ======================
import os
import wandb

print("Dang cau hinh Weights & Biases (W&B) logging...\n")

# Lay Kaggle Secret (Wandb API Key)
try:
    from kaggle_secrets import UserSecretsClient
    
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    
    if wandb_api_key:
        print("im thay WANDB_API_KEY trong Kaggle Secrets")
        
        # Login vao wandb
        wandb.login(key=wandb_api_key)
        print("Dang nhap W&B thanh cong!")
        
        # Lay thong tin user
        try:
            wandb_user = wandb.api.default_user()
            print(f"Dang nhap voi tai khoan: {wandb_user.get('username', 'Unknown')}")
        except:
            print("W&B API Key duoc xac thuc")
    else:
        print("⚠ Canh bao: Khong tim thay WANDB_API_KEY trong Kaggle Secrets")
        print("\n   Huong dan them W&B APIkey tren Kaggle:")
        print("   1. Mo Notebook Settings (top right)")
        print("   2. Chon 'Secrets' tab")
        print("   3. Them secret moi voi:")
        print("      - Key: WANDB_API_KEY")
        print("      - Value: <your-wandb-api-key>")
        print("\n   Lay W&B API key tai: https://wandb.ai/settings/api")
        print("   hoac tao tai khoan moi: https://wandb.ai/")
        print("\n   Huan luyen van se tiep tuc nhung training metrics")
        print("   se KHONG duoc log len W&B dashboard")
        
except ImportError:
    print("Canh bao: Khong the nhap UserSecretsClient")
    print("   Notebook nay can chay tren Kaggle de su dung Kaggle Secrets")
    print("   Huan luyen se su dung tensorboard thay vi wandb")
except Exception as e:
    print(f"Loi cau hinh W&B: {e}")
    print("   Huan luyen van se tiep tuc nhung khong co W&B logging")

print("\n" + "="*80)

In [ ]:
# ====================== PHAN 3: KIEM TRA DUONG DAN DATASET ======================
from datasets import load_dataset
import os

print("Dang kiem tra duong dan dataset...\n")

# Dinh nghia duong dan dataset
TRAIN_JSONL = "/kaggle/input/datasets/asiscd/nckh-vintern-dataset-jsonl/vintern_train_mixed.jsonl"
EVAL_JSONL = "/kaggle/input/datasets/asiscd/nckh-vintern-dataset-jsonl/vintern_eval.jsonl"
IMAGE_ROOT = "/kaggle/input/datasets/asiscd/nckh-vintern-dataset-images"

# Kiem tra duong dan co ton tai
for path_name, path in [
    ("JSONL Huan luyen", TRAIN_JSONL),
    ("JSONL Danh gia", EVAL_JSONL),
    ("Tung anh", IMAGE_ROOT)
]:
    if os.path.exists(path):
        print(f"Co san {path_name:20} -> {path}")
        if path.endswith('.jsonl'):
            with open(path, 'r') as f:
                num_lines = sum(1 for _ in f)
            print(f"   Chua: {num_lines} mau du lieu")
    else:
        print(f"Khong co {path_name:20} -> KHONG TIM THAY: {path}")

print("\n" + "="*80)

In [ ]:
# ====================== PHAN 4: TAI PHIEN BAN MO HINH DA HUC ======================
import os
import subprocess

print("Dang tai phien ban Vintern-1B-v3_5 tu HuggingFace...\n")

# Tao thu muc da huc
pretrained_dir = "/kaggle/working/Vintern/pretrained"
model_dir = os.path.join(pretrained_dir, "Vintern-1B-v3_5")

os.makedirs(pretrained_dir, exist_ok=True)

if os.path.exists(model_dir) and len(os.listdir(model_dir)) > 5:
    print(f"Mo hinh da duoc tai: {model_dir}")
else:
    os.chdir(pretrained_dir)
    print(f"Dang tai tu: {pretrained_dir}")
    subprocess.run([
        "huggingface-cli", "download",
        "--resume-download",
        "--local-dir-use-symlinks", "False",
        "5CD-AI/Vintern-1B-v3_5",
        "--local-dir", "Vintern-1B-v3_5"
    ], check=True)
    print("Hoan thanh tai mo hinh")

# Kiem tra tap tin mo hinh
if os.path.exists(model_dir):
    files = os.listdir(model_dir)
    print(f"\nNoi luu tru mo hinh ({len(files)} tap tin):")
    for f in sorted(files)[:5]:
        print(f"   {f}")
    if len(files) > 5:
        print(f"   ... va {len(files) - 5} tap tin khac")

print("\n" + "="*80)

In [ ]:
# ====================== PHAN 5: TAT FLASH ATTENTION (TUONG THICH KAGGLE T4) ======================
import os

print("Dang tat Flash Attention cho GPU Kaggle T4...\n")

# Duong dan tap tin flash attention patch
flash_attn_patch = "/kaggle/working/Vintern/internvl_chat/internvl/patch/llama2_flash_attn_monkey_patch.py"

# Dam bao thu muc cha ton tai
os.makedirs(os.path.dirname(flash_attn_patch), exist_ok=True)

# Tao tap tin patch moi voi Flash Attention bi tat
new_patch_content = '''# ================================================
# FLASH ATTN MONKEY PATCH - TAT CHO KAGGLE T4
# ================================================
from typing import Optional, Tuple
import torch

print(">>> Flash Attention bi tat de tuong thich GPU Kaggle")

def pad_input(*args, **kwargs):
    return args[0] if args else None

def unpad_input(*args, **kwargs):
    return args[0] if args else None

def flash_attn_func(*args, **kwargs):
    raise NotImplementedError("Flash Attention bi tat tren he thong nay")

def flash_attn_varlen_kvpacked_func(*args, **kwargs):
    raise NotImplementedError("Flash Attention bi tat tren he thong nay")

def replace_llama2_attn_with_flash_attn():
    print(">>> Flash Attention bi tat - su dung eager attention")
    return
'''

with open(flash_attn_patch, 'w', encoding='utf-8') as f:
    f.write(new_patch_content)

print(f"Hoan thanh cap nhat Flash Attention patch: {flash_attn_patch}\n")
print("="*80)

In [ ]:
# ====================== PHAN 6: SAO CHEP SCRIPT HUAN LUYEN TOI UU ======================
import os
import shutil

print("Dang sao chep script huan luyen toi uu tu tap du lieu dau vao...\n")

# Nguon: vi tri script toi uu trong tap du lieu dau vao
INPUT_SCRIPT = "/kaggle/input/datasets/asiscd/vintern-finetune/internvl_chat_finetune.py"

# Dich den: noi InternVL tiem kiem script
TARGET_DIR = "/kaggle/working/Vintern/internvl_chat/internvl/train"
TARGET_SCRIPT = os.path.join(TARGET_DIR, "internvl_chat_finetune.py")

# Tao thu muc dich neu can
os.makedirs(TARGET_DIR, exist_ok=True)

# Sao chep voi xu ly loi
try:
    if os.path.exists(INPUT_SCRIPT):
        shutil.copy(INPUT_SCRIPT, TARGET_SCRIPT)
        file_size = os.path.getsize(TARGET_SCRIPT)
        print(f"Hoan thanh sao chep script")
        print(f"   Tu: {INPUT_SCRIPT}")
        print(f"   Den: {TARGET_SCRIPT}")
        print(f"   Kich thuoc: {file_size / 1024:.1f} KB")
        
        # Kiem chung sao chep
        with open(TARGET_SCRIPT, 'r') as f:
            first_lines = [f.readline() for _ in range(3)]
            if "QLoRA" in "".join(first_lines) or "import" in first_lines[0]:
                print("   Kiem chung script: OK")
            else:
                print("   Canh bao Kiem chung script: Kiem tra thu cong")
    else:
        print(f"Khong tim thay script nguon: {INPUT_SCRIPT}")
        print("Dang kiem tra tap du lieu dau vao co san...")
        input_path = "/kaggle/input/datasets/asiscd"
        if os.path.exists(input_path):
            print(f"Co san trong {input_path}:")
            for item in os.listdir(input_path):
                print(f"  - {item}")
except Exception as e:
    print(f"Loi sao chep script: {e}")

print("\n" + "="*80)

In [ ]:
# ====================== PHAN 7: TAO TAP TIN META VUI CACH HINH DATASET ======================
import json
import os

print("Dang tao tap tin meta cho cau hinh huan luyen...\n")

# Duong dan tap tin meta
meta_dir = "/kaggle/working/Vintern/internvl_chat/shell/data"
meta_path = os.path.join(meta_dir, "custom_finetune_datasets.json")

os.makedirs(meta_dir, exist_ok=True)

# Tao cau hinh meta voi truong "length" cho hieu qua
metadata = {
    "vintern_train": {
        "annotation": "/kaggle/input/datasets/asiscd/nckh-vintern-dataset-jsonl/vintern_train_mixed.jsonl",
        "root": "/kaggle/input/datasets/asiscd/nckh-vintern-dataset-images",
        "repeat_time": 2,
        "data_augment": True,
        "length": 430  # So luong mau xap xi (tokenizer se thua tien tinh)
    },
    "vintern_eval": {
        "annotation": "/kaggle/input/datasets/asiscd/nckh-vintern-dataset-jsonl/vintern_eval.jsonl",
        "root": "/kaggle/input/datasets/asiscd/nckh-vintern-dataset-images",
        "repeat_time": 1,
        "data_augment": False,
        "length": 100  # So luong mau danh gia xap xi
    }
}

# Luu tap tin meta
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Hoan thanh tao tap tin meta: {meta_path}\n")
print("Cau hinh:")
print(json.dumps(metadata, indent=2, ensure_ascii=False))
print("\n" + "="*80)

In [ ]:
# ====================== PHAN 8: TAO SCRIPT BASH HUAN LUYEN TOI UU ======================
import os

print("Dang tao script bash huan luyen toi uu...\n")

# Duong dan script
script_dir = "/kaggle/working/Vintern/internvl_chat/shell/internvl2.0/2nd_finetune"
script_path = os.path.join(script_dir, "finetune_qlora_kaggle.sh")

os.makedirs(script_dir, exist_ok=True)

# Script huan luyen voi QLoRA duoc kich hoat va W&B logging
training_script = '''#!/bin/bash
set -x
export PYTHONPATH=$PYTHONPATH:/kaggle/working/Vintern/internvl_chat
export LAUNCHER=pytorch
export MASTER_PORT=34229

OUTPUT_DIR="/kaggle/working/work_dirs/vintern_1b_vnit_qlora"
META_PATH="/kaggle/working/Vintern/internvl_chat/shell/data/custom_finetune_datasets.json"

rm -rf "$OUTPUT_DIR"
mkdir -p "$OUTPUT_DIR"
cd /kaggle/working/Vintern

echo "=========================================="
echo "HUAN LUYEN VINTERN-1B-v3_5 VOI QLORA"
echo "=========================================="
echo "Mo hinh:    Vintern-1B-v3_5"
echo "Phuong phap: QLoRA (4-bit quantization + LoRA)"
echo "Hang LoRA: 16"
echo "GPU:       2x T4 (Kaggle)"
echo "Logging:   W&B + TensorBoard"
echo "Dau ra:    $OUTPUT_DIR"
echo "=========================================="

torchrun --nnodes=1 --node_rank=0 --master_addr=127.0.0.1 --nproc_per_node=2 --master_port=34229 \\
    internvl_chat/internvl/train/internvl_chat_finetune.py \\
    --model_name_or_path /kaggle/working/Vintern/pretrained/Vintern-1B-v3_5 \\
    --conv_style Hermes-2 \\
    --output_dir "$OUTPUT_DIR" \\
    --meta_path "$META_PATH" \\
    --overwrite_output_dir True \\
    --force_image_size 448 \\
    --max_dynamic_patch 5 \\
    --down_sample_ratio 0.5 \\
    --freeze_llm False \\
    --freeze_mlp False \\
    --freeze_backbone True \\
    --use_llm_lora 16 \\
    --use_qlora True \\
    --grad_checkpoint True \\
    --lora_alpha 16 \\
    --lora_dropout 0.05 \\
    --fp16 True \\
    --num_train_epochs 3 \\
    --per_device_train_batch_size 2 \\
    --per_device_eval_batch_size 4 \\
    --gradient_accumulation_steps 8 \\
    --evaluation_strategy steps \\
    --eval_steps 100 \\
    --save_strategy steps \\
    --save_steps 100 \\
    --save_total_limit 3 \\
    --learning_rate 1.5e-5 \\
    --weight_decay 0.02 \\
    --warmup_ratio 0.1 \\
    --lr_scheduler_type cosine \\
    --logging_steps 10 \\
    --max_seq_length 4096 \\
    --max_eval_samples 300 \\
    --do_train True \\
    --do_eval True \\
    --group_by_length False \\
    --dynamic_image_size True \\
    --use_thumbnail True \\
    --ps_version v2 \\
    --vision_select_layer -1 \\
    --deepspeed /kaggle/working/Vintern/internvl_chat/zero_stage2_config.json \\
    --dataloader_num_workers 4 \\
    --report_to wandb tensorboard \\
    --run_name "Vintern-1B-QLoRA-NCKH" \\
    2>&1 | tee -a "$OUTPUT_DIR/training_log.txt"

echo ""
echo "=========================================="
echo "Hoan thanh huan luyen!"
echo "Dau ra: $OUTPUT_DIR"
echo "=========================================="
'''

# Ghi script
with open(script_path, 'w', encoding='utf-8') as f:
    f.write(training_script)

# Lam co the thuc hien
os.chmod(script_path, 0o755)

print(f"Hoan thanh tao script huan luyen: {script_path}\n")
print("Tham so chinh:")
print("   use_qlora: True (4-bit quantization duoc kich hoat)")
print("   grad_checkpoint: True (tiet kiem bo nho)")
print("   use_llm_lora: 16 (hang LoRA)")
print("   per_device_train_batch_size: 2")
print("   num_train_epochs: 3")
print("   learning_rate: 1.5e-5")
print("   max_eval_samples: 300")
print("   report_to: wandb tensorboard (W&B + TensorBoard)")
print("   run_name: Vintern-1B-QLoRA-NCKH")
print("\n" + "="*80)

In [ ]:
# ====================== PHAN 9: KIEM TRA TIEN HUAN LUYEN ======================
import os
import json

print("Kiem tra danh sach tien huan luyen...\n")

checks = {
    "Mo hinh da tai": os.path.exists("/kaggle/working/Vintern/pretrained/Vintern-1B-v3_5/config.json"),
    "Script toi uu da sao chep": os.path.exists("/kaggle/working/Vintern/internvl_chat/internvl/train/internvl_chat_finetune.py"),
    "Tap tin meta da tao": os.path.exists("/kaggle/working/Vintern/internvl_chat/shell/data/custom_finetune_datasets.json"),
    "Script huan luyen san sang": os.path.exists("/kaggle/working/Vintern/internvl_chat/shell/internvl2.0/2nd_finetune/finetune_qlora_kaggle.sh"),
    "Flash attention bi tat": os.path.exists("/kaggle/working/Vinterns/internvl_chat/internvl/patch/llama2_flash_attn_monkey_patch.py"),
    "JSONL huan luyen co san": os.path.exists("/kaggle/input/datasets/asiscd/nckh-vintern-dataset-jsonl/vintern_train_mixed.jsonl"),
    "JSONL danh gia co san": os.path.exists("/kaggle/input/datasets/asiscd/nckh-vintern-dataset-jsonl/vintern_eval.jsonl"),
    "Thu muc anh co san": os.path.exists("/kaggle/input/datasets/asiscd/nckh-vintern-dataset-images"),
}

all_ok = True
for check_name, result in checks.items():
    status = "OK" if result else "THAT BAI"
    print(f"{'OK' if result else 'THAT BAI'}: {check_name}: {status}")
    if not result:
        all_ok = False

print("\n" + "="*80)

if all_ok:
    print("\nHoan thanh tat ca cac kiem tra! San sang huan luyen.\n")
    print("Buoc tiep theo: Chay script huan luyen (Phan 10)\n")
else:
    print("\nCanh bao: Mot so kiem tra khong thanh cong. Kiem tra lai cau hinh.\n")

print("="*80)

In [ ]:
# ====================== PHAN 10: CHAY HUAN LUYEN (CHINH) ======================
import subprocess
import os

print("\n" + "="*80)
print("BAT DAU HUAN LUYEN...")
print("="*80 + "\n")

# Duong dan script huan luyen
script_path = "/kaggle/working/Vintern/internvl_chat/shell/internvl2.0/2nd_finetune/finetune_qlora_kaggle.sh"
log_dir = "/kaggle/working/work_dirs/vintern_1b_vnit_qlora"

# Lam cho script co the thuc hien duoc
os.chmod(script_path, 0o755)

# Chay huan luyen (se mat 2-3 gio tuy vao kich thuoc dataset)
try:
    os.chdir("/kaggle/working/Vintern")
    result = subprocess.run(
        ["bash", script_path],
        capture_output=False,  # Hien thi dau ra truc tiep
        text=True,
        timeout=14400  # 4 gio timeout (gioi han phien Kaggle)
    )
    
    if result.returncode == 0:
        print("\n" + "="*80)
        print("HOAN THANH HUAN LUYEN THANH CONG!")
        print("="*80)
    else:
        print("\n" + "="*80)
        print(f"Canh bao: Huan luyen ket thuc voi ma loi: {result.returncode}")
        print("="*80)
        
except subprocess.TimeoutExpired:
    print("\n" + "="*80)
    print("Timeout huan luyen (Dat 4 gio - gioi han phien Kaggle)")
    print(f"   Kiểm tra log tai: {log_dir}")
    print("="*80)
except Exception as e:
    print(f"\nLoi huan luyen: {e}")

print(f"\nLog huan luyen da luu tai: {log_dir}")

In [ ]:
# ====================== PHAN 11: XUAT KET QUA (TUY CHON) ======================
import os
import shutil
from pathlib import Path

print("\n" + "="*80)
print("Xuat ket qua huan luyen...")
print("="*80 + "\n")

output_dir = "/kaggle/working/work_dirs/vintern_1b_vnit_qlora"

if os.path.exists(output_dir):
    # Danh sach cac thu muc checkpoint
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint")]
    print(f"Tim thay {len(checkpoints)} checkpoint:")
    for ckpt in sorted(checkpoints):
        ckpt_path = os.path.join(output_dir, ckpt)
        size = sum(f.stat().st_size for f in Path(ckpt_path).rglob('*') if f.is_file()) / (1024**3)
        print(f"  * {ckpt} ({size:.2f} GB)")
    
    # Hien thi checkpoint moi nhat
    if checkpoints:
        latest = sorted(checkpoints)[-1]
        latest_path = os.path.join(output_dir, latest)
        print(f"\nCheckpoint moi nhat: {latest}")
        print(f"   Duong dan: {latest_path}")
        
        # Kiem chung cac tap tin quan trong
        important_files = ['adapter_config.json', 'adapter_model.bin', 'pytorch_model.bin']
        print(f"   Cac tap tin:")
        for f in important_files:
            fpath = os.path.join(latest_path, f)
            if os.path.exists(fpath):
                size = os.path.getsize(fpath) / (1024**2)
                print(f"     co {f} ({size:.1f} MB)")
    
    # Log huan luyen
    log_file = os.path.join(output_dir, "training_log.txt")
    if os.path.exists(log_file):
        print(f"\nLog huan luyen co san: {log_file}")
        print("   (20 dong dau tien):")
        with open(log_file, 'r') as f:
            lines = f.readlines()[:20]
            for line in lines:
                print(f"   {line.rstrip()}")
        if len(open(log_file).readlines()) > 20:
            print(f"   ... ({len(open(log_file).readlines()) - 20} dong con lai)")
    
    # Tuy chon: Tuo chep cho tai ve
    print("\n" + "-"*80)
    print("Huong dan: De tai xuat ket qua:")
    print(f"   1. Den thu muc: {output_dir}")
    print("   2. Tai cac thu muc checkpoint va log")
    print("-"*80)
    
    # Teo tao zip cho dung sau (tuy chon)
    print("\nNhan ap: Tao tap tin nho (neu can sau nay)...")
    print("   Ban co the ZIP toan bo thu muc dau ra:")
    print(f"   zip -r model_backup.zip {output_dir}")
    
else:
    print(f"Canh bao: Khong tim thay thu muc dau ra: {output_dir}")
    print("   Huan luyen co the khong hoan thanh thanh cong.")

print("\n" + "="*80)
print("Hoan thanh xuat ket qua!")
print("="*80)

## Tài Liệu Tham Khảo: Cấu Hình Huấn Luyện & Khắc Phục Sự Cố

### Notebook Làm Điều Này

1. **Cài đặt tất cả thư viện phụ thuộc** (transformers 4.47.0, PEFT, BitsAndBytes, DeepSpeed, etc.)
2. **Nhân bản kho lưu trữ InternVL** từ GitHub (Vintern fork)
3. **Tải Phiên bản Mô hình Vintern-1B-v3_5** từ HuggingFace (khoảng 10GB)
4. **Tắt Flash Attention** để tương thích GPU Kaggle T4
5. **Sao chép tập lệnh huấn luyện tối ưu** với hỗ trợ QLoRA
6. **Tạo cấu hình metadata** cho tập dữ liệu
7. **Tạo tập lệnh bash huấn luyện** với các cờ tham số đầy đủ
8. **Chạy huấn luyện phân tán** sử dụng `torchrun` (2x T4 GPU)
9. **Xuất kết quả huấn luyện** và các điểm kiểm tra

### Cấu Hình Mô Hình

| Tham số | Giá trị | Ghi chú |
|---------|---------|--------|
| **Mô hình** | Vintern-1B-v3_5 | Mô hình Vision-Language |
| **Phương pháp** | QLoRA | 4-bit quantization + LoRA |
| **Hạng LoRA** | 16 | r=16 cho adapter |
| **LoRA Alpha** | 16 | hệ số điều chỉnh |
| **Kích thước Ảnh** | 448x448 | force_image_size |
| **Kiểu Cuộc Thoại** | Hermes-2 | định dạng cuộc thoại |
| **Chiều dài Chuỗi Tối đa** | 4096 | chiều dài chuỗi |
| **Kích Thước Theo nhóm** | 2 | trên mỗi thiết bị |
| **Thời Kỳ Huấn luyện** | 3 | số lượng epoch |
| **Tỷ Lệ Học** | 1.5e-5 | LR ban đầu |
| **Lịch Học Tập** | Cosine | với khởi động 0.1 |

### Các Cờ Hướng dẫn Huấn luyện Chính

```bash
--use_qlora True              # Kích hoạt QLoRA (4-bit quantization)
--grad_checkpoint True        # Gradient checkpointing để tiết kiệm bộ nhớ
--freeze_backbone True        # Đóng băng thị giác encoder
--freeze_llm False            # Tinh chỉnh tham số LLM (qua LoRA adapter)
--freeze_mlp False            # Tinh chỉnh MLP projector
--dynamic_image_size True     # Cho phép các độ phân giải ảnh khác nhau
--max_eval_samples 300        # Giới hạn đánh giá đến 300 mẫu
--do_eval True                # Kích hoạt đánh giá trong quá trình huấn luyện
```

### Tài Nguyên Cần Thiết

| Phạm Vi | Giá Trị |
|---------|---------|
| **Bộ Nhớ GPU** | khoảng 8-9 GB cho mỗi T4 (với QLoRA) |
| **Thời Gian Huấn luyện** | 2-3 giờ (2 epoch, 2x T4) |
| **Kích Thước Mô hình (LoRA)** | khoảng 50-100 MB cho mỗi điểm kiểm tra |
| **Không Gian Checkpoint** | khoảng 300-500 MB tổng cộng |

### Khắc Phục Sự Cố

**Vấn đề**: Lỗi Hết Bộ Nhớ (OOM)
- Giảm `per_device_train_batch_size` xuống 1
- Tăng `gradient_accumulation_steps` lên 16
- Kích hoạt `--grad_checkpoint True` (đã kích hoạt)

**Vấn đề**: Tập dữ liệu không tìm thấy
- Kiểm chứng tập dữ liệu nằm trong đầu vào: `/kaggle/input/datasets/asiscd/nckh-vintern-dataset-*`
- Kiểm tra đường dẫn tệp trong tệp meta JSON

**Vấn đề**: Sao chép tập lệnh thất bại
- Đặt tệp vào Thư mục `/kaggle/working/Vintern/internvl_chat/internvl/train/`
- Hoặc trích xuất từ tập dữ liệu đầu vào: `/kaggle/input/datasets/asiscd/vintern-finetune/`

**Vấn đề**: Lỗi Flash Attention
- Đã bị tắt trong Phần 5
- Sử dụng eager attention fallback (chậm hơn một chút nhưng ổn định)

### Các Bước Tiếp Theo Sau Khi Huấn luyện

1. **Tải mô hình đã huấn luyện** từ đầu ra Kaggle
2. **Hợp nhất LoRA adapter** với mô hình cơ bản (tuỳ chọn)
3. **Kiểm tra trên dữ liệu mới** sử dụng notebook suy luận
4. **Đẩy lên HuggingFace Hub** (tuỳ chọn)

### Tài Liệu Liên Quan

- [Vintern GitHub](https://github.com/5CD-AI/Vintern)
- [InternVL Docs](https://github.com/OpenGVLab/InternVL)
- [PEFT Documentation](https://huggingface.co/docs/peft)
- [QLoRA Paper](https://arxiv.org/abs/2305.14314)